# Optional Cleanup - Delete All Resources

⚠️ **WARNING: This notebook permanently deletes AWS resources. Use with caution!**

This notebook helps you clean up ALL resources created by this project:
- SageMaker endpoints, models, and configurations
- Lambda functions and event schedules
- SQS queues and SNS topics
- CloudWatch dashboards and alarms
- Athena tables
- S3 data (optional)
- IAM roles (optional)
- ECR repositories (optional)

**Before running this notebook:**
1. ✅ Export any data you want to keep
2. ✅ Document your configuration (thresholds, endpoints, etc.)
3. ✅ Review what will be deleted
4. ✅ Confirm you want to proceed

**Sections:**
- Setup
- 1. Drift Monitoring Infrastructure
- 2. SageMaker Resources (Endpoints, Models)
- 3. Athena Tables
- 4. CloudWatch Resources
- 5. S3 Data (Optional)
- 6. IAM Roles (Optional)
- 7. Summary

## Setup

In [ ]:
import sys
import os
from pathlib import Path
import boto3
import time

# Find project root
project_root = Path.cwd()
while not (project_root / '.env').exists() and project_root != project_root.parent:
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(project_root / '.env')

from src.config.config import (
    ATHENA_DATABASE, DATA_S3_BUCKET, AWS_DEFAULT_REGION
)

# AWS clients
REGION = AWS_DEFAULT_REGION
sts = boto3.client('sts', region_name=REGION)
AWS_ACCOUNT_ID = sts.get_caller_identity()['Account']

print(f"Region: {REGION}")
print(f"Account ID: {AWS_ACCOUNT_ID}")
print(f"S3 Bucket: {DATA_S3_BUCKET}")
print(f"Athena DB: {ATHENA_DATABASE}")
print("")
print("⚠️  Ready to delete resources. Review each section before running.")

---

## 1. Drift Monitoring Infrastructure

Delete Lambda functions, EventBridge rules, SQS queues, SNS topics, and ECR repositories.

### 1.1 List Drift Monitoring Resources

In [ ]:
# Get resource names from .env
DRIFT_LAMBDA = os.getenv('DRIFT_LAMBDA_NAME', 'fraud-detection-drift-monitor')
MONITORING_WRITER_LAMBDA = os.getenv('MONITORING_WRITER_LAMBDA_NAME', 'fraud-monitoring-results-writer')
SNS_TOPIC_NAME = os.getenv('SNS_TOPIC_NAME', 'fraud-detection-drift-alerts')
MONITORING_SQS_QUEUE = os.getenv('MONITORING_SQS_QUEUE_NAME', 'fraud-monitoring-results')
EVENTBRIDGE_RULE = os.getenv('EVENTBRIDGE_RULE_NAME', 'fraud-detection-drift-check')

print("Drift Monitoring Resources:")
print("="*80)

# Check Lambda functions
lambda_client = boto3.client('lambda', region_name=REGION)
for lambda_name in [DRIFT_LAMBDA, MONITORING_WRITER_LAMBDA]:
    try:
        resp = lambda_client.get_function(FunctionName=lambda_name)
        print(f"✓ Lambda: {lambda_name}")
    except lambda_client.exceptions.ResourceNotFoundException:
        print(f"  (Lambda {lambda_name} not found)")

# Check EventBridge rule
events = boto3.client('events', region_name=REGION)
try:
    rule = events.describe_rule(Name=EVENTBRIDGE_RULE)
    print(f"✓ EventBridge Rule: {EVENTBRIDGE_RULE}")
except events.exceptions.ResourceNotFoundException:
    print(f"  (EventBridge rule {EVENTBRIDGE_RULE} not found)")

# Check SQS queue
sqs = boto3.client('sqs', region_name=REGION)
try:
    queue_url = sqs.get_queue_url(QueueName=MONITORING_SQS_QUEUE)['QueueUrl']
    print(f"✓ SQS Queue: {MONITORING_SQS_QUEUE}")
except sqs.exceptions.QueueDoesNotExist:
    print(f"  (SQS queue {MONITORING_SQS_QUEUE} not found)")

# Check SNS topic
sns_client = boto3.client('sns', region_name=REGION)
topics = sns_client.list_topics()['Topics']
found_topic = [t for t in topics if SNS_TOPIC_NAME in t['TopicArn']]
if found_topic:
    print(f"✓ SNS Topic: {SNS_TOPIC_NAME}")
else:
    print(f"  (SNS topic {SNS_TOPIC_NAME} not found)")

# Check ECR repository
ecr = boto3.client('ecr', region_name=REGION)
try:
    repos = ecr.describe_repositories(repositoryNames=[DRIFT_LAMBDA])
    print(f"✓ ECR Repository: {DRIFT_LAMBDA}")
except ecr.exceptions.RepositoryNotFoundException:
    print(f"  (ECR repository {DRIFT_LAMBDA} not found)")

print("="*80)

### 1.2 Delete Drift Monitoring Infrastructure

**Using the delete_infrastructure.sh script** (recommended):

In [ ]:
import subprocess

# Set to 'yes' to also delete ECR images
DELETE_ECR = 'no'  # Change to 'yes' if you want to delete ECR repository too

print("Running delete_infrastructure.sh script...")
print("="*80)

result = subprocess.run(
    ['bash', 'scripts/delete_infrastructure.sh', DELETE_ECR],
    cwd=project_root,
    input='yes\n',  # Auto-confirm deletion
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print("Errors:")
    print(result.stderr)

if result.returncode == 0:
    print("\n✓ Drift monitoring infrastructure deleted successfully")
else:
    print(f"\n⚠ Script exited with code {result.returncode}")

---

## 2. SageMaker Resources

Delete endpoints, models, and endpoint configurations.

### 2.1 List SageMaker Resources

In [ ]:
sm_client = boto3.client('sagemaker', region_name=REGION)

print("SageMaker Resources:")
print("="*80)

# List endpoints
endpoints = sm_client.list_endpoints(MaxResults=100)['Endpoints']
fraud_endpoints = [e for e in endpoints if 'fraud' in e['EndpointName'].lower()]

if fraud_endpoints:
    print(f"\nEndpoints ({len(fraud_endpoints)}):")
    for ep in fraud_endpoints:
        print(f"  - {ep['EndpointName']} (Status: {ep['EndpointStatus']})")
else:
    print("\n  (No fraud detection endpoints found)")

# List models
models = sm_client.list_models(MaxResults=100)['Models']
fraud_models = [m for m in models if 'fraud' in m['ModelName'].lower()]

if fraud_models:
    print(f"\nModels ({len(fraud_models)}):")
    for model in fraud_models:
        print(f"  - {model['ModelName']}")
else:
    print("\n  (No fraud detection models found)")

# List endpoint configs
configs = sm_client.list_endpoint_configs(MaxResults=100)['EndpointConfigs']
fraud_configs = [c for c in configs if 'fraud' in c['EndpointConfigName'].lower()]

if fraud_configs:
    print(f"\nEndpoint Configs ({len(fraud_configs)}):")
    for config in fraud_configs:
        print(f"  - {config['EndpointConfigName']}")
else:
    print("\n  (No fraud detection endpoint configs found)")

print("="*80)

### 2.2 Delete SageMaker Endpoints

In [ ]:
# Delete endpoints
for ep in fraud_endpoints:
    endpoint_name = ep['EndpointName']
    try:
        print(f"Deleting endpoint: {endpoint_name}...")
        sm_client.delete_endpoint(EndpointName=endpoint_name)
        print(f"  ✓ Endpoint {endpoint_name} deletion initiated")
    except Exception as e:
        print(f"  ✗ Error deleting {endpoint_name}: {e}")

if fraud_endpoints:
    print(f"\n✓ Deleted {len(fraud_endpoints)} endpoints")
else:
    print("\n  (No endpoints to delete)")

### 2.3 Delete SageMaker Endpoint Configs

In [ ]:
# Delete endpoint configs
for config in fraud_configs:
    config_name = config['EndpointConfigName']
    try:
        print(f"Deleting endpoint config: {config_name}...")
        sm_client.delete_endpoint_config(EndpointConfigName=config_name)
        print(f"  ✓ Endpoint config {config_name} deleted")
    except Exception as e:
        print(f"  ✗ Error deleting {config_name}: {e}")

if fraud_configs:
    print(f"\n✓ Deleted {len(fraud_configs)} endpoint configs")
else:
    print("\n  (No endpoint configs to delete)")

### 2.4 Delete SageMaker Models

In [ ]:
# Delete models
for model in fraud_models:
    model_name = model['ModelName']
    try:
        print(f"Deleting model: {model_name}...")
        sm_client.delete_model(ModelName=model_name)
        print(f"  ✓ Model {model_name} deleted")
    except Exception as e:
        print(f"  ✗ Error deleting {model_name}: {e}")

if fraud_models:
    print(f"\n✓ Deleted {len(fraud_models)} models")
else:
    print("\n  (No models to delete)")

---

## 3. Athena Tables

Delete Athena tables used for training data, inference logging, and monitoring.

### 3.1 List Athena Tables

In [ ]:
athena = boto3.client('athena', region_name=REGION)

print(f"Athena Tables in {ATHENA_DATABASE}:")
print("="*80)

# Get tables
query = f"SHOW TABLES IN {ATHENA_DATABASE}"
response = athena.start_query_execution(
    QueryString=query,
    QueryExecutionContext={'Database': ATHENA_DATABASE},
    ResultConfiguration={
        'OutputLocation': f's3://{DATA_S3_BUCKET}/athena-query-results/'
    }
)

execution_id = response['QueryExecutionId']

# Wait for query
while True:
    status = athena.get_query_execution(QueryExecutionId=execution_id)
    state = status['QueryExecution']['Status']['State']
    if state in ['SUCCEEDED', 'FAILED', 'CANCELLED']:
        break
    time.sleep(1)

if state == 'SUCCEEDED':
    results = athena.get_query_results(QueryExecutionId=execution_id)
    tables = [row['Data'][0]['VarCharValue'] for row in results['ResultSet']['Rows'][1:]]
    
    if tables:
        print("\nTables found:")
        for table in tables:
            print(f"  - {table}")
    else:
        print("\n  (No tables found)")
else:
    print(f"\n  Query failed: {state}")
    tables = []

print("="*80)

### 3.2 Delete Athena Tables

⚠️ **WARNING: This permanently deletes table metadata. S3 data remains unless deleted separately.**

In [ ]:
# Tables to delete
TABLES_TO_DELETE = [
    'training_data',
    'inference_responses',
    'ground_truth',
    'ground_truth_updates',
    'drifted_data',
    'monitoring_responses'
]

# Confirm deletion
CONFIRM_DELETE_TABLES = False  # Change to True to enable deletion

if not CONFIRM_DELETE_TABLES:
    print("⚠️  Set CONFIRM_DELETE_TABLES = True to enable deletion")
else:
    for table_name in TABLES_TO_DELETE:
        if table_name in tables:
            try:
                print(f"Dropping table: {table_name}...")
                query = f"DROP TABLE IF EXISTS {ATHENA_DATABASE}.{table_name}"
                response = athena.start_query_execution(
                    QueryString=query,
                    QueryExecutionContext={'Database': ATHENA_DATABASE},
                    ResultConfiguration={
                        'OutputLocation': f's3://{DATA_S3_BUCKET}/athena-query-results/'
                    }
                )
                
                # Wait for query
                execution_id = response['QueryExecutionId']
                while True:
                    status = athena.get_query_execution(QueryExecutionId=execution_id)
                    state = status['QueryExecution']['Status']['State']
                    if state in ['SUCCEEDED', 'FAILED', 'CANCELLED']:
                        break
                    time.sleep(1)
                
                if state == 'SUCCEEDED':
                    print(f"  ✓ Table {table_name} dropped")
                else:
                    print(f"  ✗ Failed to drop {table_name}: {state}")
                    
            except Exception as e:
                print(f"  ✗ Error dropping {table_name}: {e}")
    
    print(f"\n✓ Table deletion complete")

---

## 4. CloudWatch Resources

Delete CloudWatch dashboards and alarms.

### 4.1 List CloudWatch Resources

In [ ]:
cw = boto3.client('cloudwatch', region_name=REGION)

print("CloudWatch Resources:")
print("="*80)

# List dashboards
dashboards = cw.list_dashboards()['DashboardEntries']
fraud_dashboards = [d for d in dashboards if 'fraud' in d['DashboardName'].lower()]

if fraud_dashboards:
    print(f"\nDashboards ({len(fraud_dashboards)}):")
    for dash in fraud_dashboards:
        print(f"  - {dash['DashboardName']}")
else:
    print("\n  (No fraud detection dashboards found)")

# List alarms
alarms = cw.describe_alarms()['MetricAlarms']
fraud_alarms = [a for a in alarms if 'fraud' in a['AlarmName'].lower()]

if fraud_alarms:
    print(f"\nAlarms ({len(fraud_alarms)}):")
    for alarm in fraud_alarms:
        print(f"  - {alarm['AlarmName']} (State: {alarm['StateValue']})")
else:
    print("\n  (No fraud detection alarms found)")

print("="*80)

### 4.2 Delete CloudWatch Dashboards

In [ ]:
for dash in fraud_dashboards:
    dashboard_name = dash['DashboardName']
    try:
        print(f"Deleting dashboard: {dashboard_name}...")
        cw.delete_dashboards(DashboardNames=[dashboard_name])
        print(f"  ✓ Dashboard {dashboard_name} deleted")
    except Exception as e:
        print(f"  ✗ Error deleting {dashboard_name}: {e}")

if fraud_dashboards:
    print(f"\n✓ Deleted {len(fraud_dashboards)} dashboards")
else:
    print("\n  (No dashboards to delete)")

### 4.3 Delete CloudWatch Alarms

In [ ]:
if fraud_alarms:
    alarm_names = [a['AlarmName'] for a in fraud_alarms]
    try:
        print(f"Deleting {len(alarm_names)} alarms...")
        cw.delete_alarms(AlarmNames=alarm_names)
        for name in alarm_names:
            print(f"  ✓ Alarm {name} deleted")
        print(f"\n✓ Deleted {len(alarm_names)} alarms")
    except Exception as e:
        print(f"  ✗ Error deleting alarms: {e}")
else:
    print("\n  (No alarms to delete)")

---

## 5. S3 Data (Optional)

⚠️ **WARNING: This permanently deletes all S3 data. Cannot be undone!**

Only run this if you're sure you don't need any of the data.

### 5.1 List S3 Data

In [ ]:
s3 = boto3.client('s3', region_name=REGION)

print(f"S3 Data in bucket: {DATA_S3_BUCKET}")
print("="*80)

# List prefixes
prefixes_to_check = [
    'fraud-detection/',
    'athena-query-results/',
    'sagemaker-fraud-detection/',
    'model_artifacts/',
]

total_size = 0
total_objects = 0

for prefix in prefixes_to_check:
    try:
        paginator = s3.get_paginator('list_objects_v2')
        pages = paginator.paginate(Bucket=DATA_S3_BUCKET, Prefix=prefix)
        
        prefix_size = 0
        prefix_count = 0
        
        for page in pages:
            if 'Contents' in page:
                for obj in page['Contents']:
                    prefix_size += obj['Size']
                    prefix_count += 1
        
        if prefix_count > 0:
            size_mb = prefix_size / (1024 * 1024)
            print(f"  {prefix}: {prefix_count} objects, {size_mb:.2f} MB")
            total_size += prefix_size
            total_objects += prefix_count
            
    except Exception as e:
        print(f"  Error checking {prefix}: {e}")

if total_objects > 0:
    total_mb = total_size / (1024 * 1024)
    total_gb = total_size / (1024 * 1024 * 1024)
    print(f"\nTotal: {total_objects} objects, {total_mb:.2f} MB ({total_gb:.3f} GB)")
else:
    print("\n  (No S3 objects found)")

print("="*80)

### 5.2 Delete S3 Data

⚠️ **DANGER ZONE: Set CONFIRM_DELETE_S3_DATA = True to enable deletion**

In [ ]:
CONFIRM_DELETE_S3_DATA = False  # Change to True to enable S3 deletion

if not CONFIRM_DELETE_S3_DATA:
    print("⚠️  Set CONFIRM_DELETE_S3_DATA = True to enable S3 deletion")
    print("⚠️  This will permanently delete ALL data. Cannot be undone!")
else:
    print("⚠️  DELETING S3 DATA...")
    print("="*80)
    
    for prefix in prefixes_to_check:
        try:
            print(f"\nDeleting objects with prefix: {prefix}...")
            
            # List and delete in batches
            paginator = s3.get_paginator('list_objects_v2')
            pages = paginator.paginate(Bucket=DATA_S3_BUCKET, Prefix=prefix)
            
            deleted_count = 0
            for page in pages:
                if 'Contents' in page:
                    objects = [{'Key': obj['Key']} for obj in page['Contents']]
                    if objects:
                        s3.delete_objects(
                            Bucket=DATA_S3_BUCKET,
                            Delete={'Objects': objects}
                        )
                        deleted_count += len(objects)
            
            if deleted_count > 0:
                print(f"  ✓ Deleted {deleted_count} objects from {prefix}")
            else:
                print(f"  (No objects found in {prefix})")
                
        except Exception as e:
            print(f"  ✗ Error deleting {prefix}: {e}")
    
    print("\n✓ S3 data deletion complete")

---

## 6. IAM Roles (Optional)

⚠️ **WARNING: Only delete roles if you're sure they're not used by other resources.**

### 6.1 List IAM Roles

In [ ]:
iam = boto3.client('iam', region_name=REGION)

print("IAM Roles (fraud-related):")
print("="*80)

# List roles
roles = iam.list_roles()['Roles']
fraud_roles = [r for r in roles if 'fraud' in r['RoleName'].lower()]

if fraud_roles:
    print("\nRoles found:")
    for role in fraud_roles:
        print(f"  - {role['RoleName']}")
else:
    print("\n  (No fraud detection roles found)")

print("="*80)
print("\n⚠️  Manual deletion recommended. Use AWS Console or CLI to review and delete.")
print("⚠️  Do NOT delete roles that might be used by other services.")

---

## 7. Summary

Review what was deleted.

In [ ]:
print("="*80)
print("                    CLEANUP SUMMARY")
print("="*80)
print("")
print("Resources deleted:")
print("")
print("✓ Drift Monitoring:")
print("  - Lambda functions (drift monitor, monitoring writer)")
print("  - EventBridge rules")
print("  - SQS queues")
print("  - SNS topics")
print("  - ECR repositories (if enabled)")
print("")
print("✓ SageMaker:")
print(f"  - Endpoints: {len(fraud_endpoints)}")
print(f"  - Models: {len(fraud_models)}")
print(f"  - Endpoint Configs: {len(fraud_configs)}")
print("")
print("✓ CloudWatch:")
print(f"  - Dashboards: {len(fraud_dashboards)}")
print(f"  - Alarms: {len(fraud_alarms)}")
print("")
if 'CONFIRM_DELETE_TABLES' in locals() and CONFIRM_DELETE_TABLES:
    print("✓ Athena Tables: Deleted")
else:
    print("⚠ Athena Tables: Not deleted (set CONFIRM_DELETE_TABLES = True)")
print("")
if 'CONFIRM_DELETE_S3_DATA' in locals() and CONFIRM_DELETE_S3_DATA:
    print("✓ S3 Data: Deleted")
else:
    print("⚠ S3 Data: Not deleted (set CONFIRM_DELETE_S3_DATA = True)")
print("")
print("⚠ IAM Roles: Review and delete manually via AWS Console")
print("")
print("="*80)
print("")
print("Next steps:")
print("  1. Verify resources are deleted in AWS Console")
print("  2. Check for any remaining S3 objects")
print("  3. Review CloudWatch logs if needed")
print("  4. Delete IAM roles manually if not needed")
print("")
print("✅ Cleanup complete!")
print("="*80)